# Steam Crawler Colab Workflow

Run the cells from top to bottom. If anything is unclear or breaks, check `README.md` in the project folder for the fuller setup and workflow notes.


In [ ]:
from __future__ import annotations

from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

# Update this path if your project folder lives somewhere else in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/steam_crawler')
SRC_DIR = PROJECT_DIR / 'src' / 'steam_crawler'
DATA_DIR = PROJECT_DIR / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = DATA_DIR / 'figures'
PROJECT_DIR


In [ ]:
required_files = [
    SRC_DIR / 'crawler.py',
    SRC_DIR / 'preprocess_positioning.py',
    SRC_DIR / 'aggregate_taxonomy_matrix.py',
    SRC_DIR / 'run_taxonomy_pca.py',
    SRC_DIR / 'run_taxonomy_umap.py',
    SRC_DIR / 'annotate_taxonomy_umap.py',
    SRC_DIR / 'annotate_famous_games_umap.py',
    PROJECT_DIR / 'requirements.txt',
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        'PROJECT_DIR does not look like the steam_crawler folder. Missing: ' + ', '.join(missing)
    )

print('Project folder found:', PROJECT_DIR)
sorted(path.name for path in PROJECT_DIR.iterdir())[:12]


In [ ]:
%cd /content
!python -m pip install --upgrade pip
!python -m pip install -r "$PROJECT_DIR/requirements.txt"


In [ ]:
import json
import shutil
import subprocess
from IPython.display import Image, display
import pandas as pd


def run_command(args: list[str], cwd: Path = PROJECT_DIR) -> None:
    print('$', ' '.join(args))
    completed = subprocess.run(args, cwd=str(cwd), check=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)


def show_png(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path)))


## Step 1 - Choose an input mode

For most Colab runs, reusing an existing sample JSON is the smoothest option. Live crawling also works, but it takes longer and depends on external APIs staying responsive during the session.


In [ ]:
# Recommended for Colab: reuse a sample that already exists in the project folder.
USE_EXISTING_SAMPLE = True
EXISTING_SAMPLE_JSON = RAW_DATA_DIR / 'sample_100.json'

# If you want to crawl live data instead, set USE_EXISTING_SAMPLE = False.
LIVE_JSON_OUT = RAW_DATA_DIR / 'colab_sample.json'
LIVE_CSV_OUT = RAW_DATA_DIR / 'colab_sample.csv'

if USE_EXISTING_SAMPLE:
    input_json = EXISTING_SAMPLE_JSON
    if not input_json.exists():
        raise FileNotFoundError(f'Existing sample not found: {input_json}')
else:
    run_command([
        'python',
        str(SRC_DIR / 'crawler.py'),
        '--sample-size', '100',
        '--sample-seed', '42',
        '--steamspy-catalog-pages', '3',
        '--steamspy-page-delay', '15',
        '--delay', '1.0',
        '--progress-every', '20',
        '--json-out', str(LIVE_JSON_OUT),
        '--csv-out', str(LIVE_CSV_OUT),
    ])
    input_json = LIVE_JSON_OUT

print('Using input JSON:', input_json)


In [ ]:
sample_rows = json.loads(input_json.read_text(encoding='utf-8'))
print('Rows in selected input:', len(sample_rows))
sample_rows[0]


## Step 2 - Preprocess the crawled data

This turns the raw JSON into analysis-friendly tables such as `clean_games.csv`, `tag_stats.csv`, and `tag_matrix.csv`.


In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

run_command([
    'python',
    str(SRC_DIR / 'preprocess_positioning.py'),
    '--input', str(input_json),
    '--out-dir', str(PROCESSED_DIR),
])


In [ ]:
clean_games = pd.read_csv(PROCESSED_DIR / 'clean_games.csv', encoding='utf-8-sig')
tag_stats = pd.read_csv(PROCESSED_DIR / 'tag_stats.csv', encoding='utf-8-sig')

print('clean_games shape:', clean_games.shape)
print('tag_stats shape:', tag_stats.shape)
display(clean_games.head(5))
display(tag_stats.head(10))


## Step 3 - Build the taxonomy matrix

This step merges individual Steam tags into higher-level taxonomy features using `taxonomy_mapping.csv`.


In [ ]:
taxonomy_mapping = PROCESSED_DIR / 'taxonomy_mapping.csv'
if not taxonomy_mapping.exists():
    raise FileNotFoundError(
        'taxonomy_mapping.csv was not found at the expected path: ' + str(taxonomy_mapping)
    )

run_command([
    'python',
    str(SRC_DIR / 'aggregate_taxonomy_matrix.py'),
    '--tag-matrix', str(PROCESSED_DIR / 'tag_matrix.csv'),
    '--mapping', str(taxonomy_mapping),
    '--out', str(PROCESSED_DIR / 'taxonomy_matrix.csv'),
    '--feature-out', str(PROCESSED_DIR / 'taxonomy_features.csv'),
])


In [ ]:
taxonomy_matrix = pd.read_csv(PROCESSED_DIR / 'taxonomy_matrix.csv', encoding='utf-8-sig')
taxonomy_features = pd.read_csv(PROCESSED_DIR / 'taxonomy_features.csv', encoding='utf-8-sig')

print('taxonomy_matrix shape:', taxonomy_matrix.shape)
print('taxonomy_features shape:', taxonomy_features.shape)
display(taxonomy_matrix.head(5))
display(taxonomy_features.head(10))


## Step 4 - Run PCA and UMAP

These two scripts create lower-dimensional positioning maps from the taxonomy matrix.


In [ ]:
run_command([
    'python',
    str(SRC_DIR / 'run_taxonomy_pca.py'),
    '--input', str(PROCESSED_DIR / 'taxonomy_matrix.csv'),
    '--coords-out', str(PROCESSED_DIR / 'taxonomy_pca_coords.csv'),
    '--plot-out', str(FIGURES_DIR / 'taxonomy_pca_scatter.png'),
    '--loadings-out', str(PROCESSED_DIR / 'taxonomy_pca_loadings.csv'),
    '--plot3d-out', str(FIGURES_DIR / 'taxonomy_pca_scatter_3d.png'),
])

run_command([
    'python',
    str(SRC_DIR / 'run_taxonomy_umap.py'),
    '--input', str(PROCESSED_DIR / 'taxonomy_matrix.csv'),
    '--coords-out', str(PROCESSED_DIR / 'taxonomy_umap_coords.csv'),
    '--plot-out', str(FIGURES_DIR / 'taxonomy_umap_scatter.png'),
])


In [ ]:
show_png(FIGURES_DIR / 'taxonomy_pca_scatter.png')
show_png(FIGURES_DIR / 'taxonomy_umap_scatter.png')


## Step 5 - Annotate the UMAP map

Now we add content-cluster labels and mark the most famous or top-selling games.


In [ ]:
run_command([
    'python',
    str(SRC_DIR / 'annotate_taxonomy_umap.py'),
    '--umap-coords', str(PROCESSED_DIR / 'taxonomy_umap_coords.csv'),
    '--taxonomy-matrix', str(PROCESSED_DIR / 'taxonomy_matrix.csv'),
    '--plot-out', str(FIGURES_DIR / 'taxonomy_umap_annotated.png'),
    '--summary-out', str(PROCESSED_DIR / 'taxonomy_cluster_summary.csv'),
    '--games-out', str(PROCESSED_DIR / 'taxonomy_umap_clusters.csv'),
])

run_command([
    'python',
    str(SRC_DIR / 'annotate_famous_games_umap.py'),
    '--coords', str(PROCESSED_DIR / 'taxonomy_umap_coords.csv'),
    '--clean-games', str(PROCESSED_DIR / 'clean_games.csv'),
    '--plot-out', str(FIGURES_DIR / 'taxonomy_umap_top_games.png'),
    '--labels-out', str(PROCESSED_DIR / 'top_games_labels.csv'),
    '--top-n', '20',
])


In [ ]:
show_png(FIGURES_DIR / 'taxonomy_umap_annotated.png')
show_png(FIGURES_DIR / 'taxonomy_umap_top_games.png')

cluster_summary = pd.read_csv(PROCESSED_DIR / 'taxonomy_cluster_summary.csv', encoding='utf-8-sig')
top_games = pd.read_csv(PROCESSED_DIR / 'top_games_labels.csv', encoding='utf-8-sig')
display(cluster_summary)
display(top_games.head(20))


## Step 6 - Optional interactive 3D PCA biplot

This step creates an HTML file with interactive rotation and hover tooltips. In Colab, it is often easiest to generate the HTML and then open or download it.


In [ ]:
run_command([
    'python',
    str(SRC_DIR / 'plot_taxonomy_pca_biplot_3d_interactive.py'),
    '--input', str(PROCESSED_DIR / 'taxonomy_matrix.csv'),
    '--clean-games', str(PROCESSED_DIR / 'clean_games.csv'),
    '--html-out', str(FIGURES_DIR / 'taxonomy_pca_biplot_3d.html'),
])

print('Interactive HTML written to:', FIGURES_DIR / 'taxonomy_pca_biplot_3d.html')


## Step 7 - Export the results

This bundles the Colab outputs into one zip file that you can download or share from Drive.


In [ ]:
processed_archive = shutil.make_archive(str(DATA_DIR / 'colab_processed_bundle'), 'zip', root_dir=str(PROCESSED_DIR))
figures_archive = shutil.make_archive(str(DATA_DIR / 'colab_figures_bundle'), 'zip', root_dir=str(FIGURES_DIR))
print('Created archive:', processed_archive)
print('Created archive:', figures_archive)


## Notes

- If `PROJECT_DIR` is wrong, the validation cell near the top will fail early.
- Reusing an existing sample JSON is usually more stable in Colab than doing a fresh live crawl.
- If you change `taxonomy_mapping.csv`, rerun aggregation, PCA, UMAP, and the annotation steps.
- If you need more context on any step, open `README.md` in the project folder.
